In [2]:
# ============================================================================
# CELL 1: Imports
# ============================================================================

import re
import os
import time
import warnings

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")
print("Libraries imported.")

Libraries imported.


In [3]:
# ============================================================================
# CELL 2: Configuration and LLM transport
# ============================================================================
import os, time, requests

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
from dotenv import load_dotenv

assert load_dotenv("Key.env"), "Key.env not found — check it's beside the notebook"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY not set in Key.env"

MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

# Whether the model may do arithmetic, or must quote figures exactly as supplied.
# False during development: a weak model's math errors and its grounding failures look
# identical in the output, and only one of those is what we're testing.
ALLOW_MODEL_ARITHMETIC = False

LLM_AVAILABLE = False
HEADERS  = {"Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"}
RETRYABLE = {408, 429, 500, 502, 503, 504}


class _Retry(Exception):
    def __init__(self, code, msg):
        self.code, self.msg = code, msg
        super().__init__(f"{code}: {msg}")


def _unwrap(r):
    """Extract content, or raise. OpenRouter returns 200 with an error body when the
    upstream provider fails — trusting the status line gives a KeyError three frames down."""
    payload = r.json()
    if "error" in payload:
        err, code = payload["error"], payload["error"].get("code", r.status_code)
        if code in RETRYABLE:
            raise _Retry(code, err.get("message", ""))
        raise RuntimeError(f"provider error {code}: {err.get('message', '')[:200]}")
    if not payload.get("choices"):
        raise RuntimeError(f"no choices: {str(payload)[:300]}")

    u = payload.get("usage", {})
    reasoning = (u.get("completion_tokens_details") or {}).get("reasoning_tokens", 0)
    print(f"  tokens: {u.get('completion_tokens', '?')} completion"
          f"{f' ({reasoning} reasoning)' if reasoning else ''}"
          f" | finish: {payload['choices'][0].get('finish_reason')}")

    # A truncated reply is not a shorter reply — it cuts mid-figure, and the checker then
    # flags the fragment as an unsupported number. Fail loudly rather than return it.
    if payload["choices"][0].get("finish_reason") == "length":
        raise RuntimeError(
            f"truncated at max_tokens ({u.get('completion_tokens')} used, "
            f"{reasoning} of them reasoning) — raise max_tokens or use a non-reasoning model")

    return payload["choices"][0]["message"]["content"]


def call_llm(system, user, max_tokens=2000, temperature=0.0, retries=3):
    """Single transport for every LLM call. `system` is a parameter, not a constant —
    a hardcoded persona leaves the citation rules nowhere to live."""
    for attempt in range(retries):
        try:
            r = requests.post(OPENROUTER_URL, headers=HEADERS, timeout=120,
                              json={"model": MODEL, "temperature": temperature,
                                    "max_tokens": max_tokens,
                                    "reasoning": {"enabled": False},
                                    "messages": [{"role": "system", "content": system},
                                                 {"role": "user", "content": user}]})
            if r.status_code in RETRYABLE:
                raise _Retry(r.status_code, r.text[:200])
            return _unwrap(r)
        except _Retry as e:
            wait = 15 * (attempt + 1)
            print(f"  retryable ({e.code}), waiting {wait}s — {e.msg[:120]}")
            time.sleep(wait)
    raise RuntimeError(f"LLM unavailable after {retries} attempts")


def check_llm():
    """Verify the endpoint. Sets LLM_AVAILABLE; never raises."""
    global LLM_AVAILABLE
    try:
        # 200, not 5: a reasoning model spends tokens before emitting anything, so a
        # tight ceiling truncates the probe rather than testing the connection.
        call_llm("Reply with exactly: ok", "ping", max_tokens=200, retries=1)
        LLM_AVAILABLE = True
        print(f"LLM ready: {MODEL} | model arithmetic: "
              f"{'permitted' if ALLOW_MODEL_ARITHMETIC else 'disabled'}")
    except Exception as e:
        LLM_AVAILABLE = False
        print(f"LLM unavailable — {type(e).__name__}: {str(e)[:160]}")
        print("The system will work without the LLM and answer from the data directly.")
    return LLM_AVAILABLE

    

check_llm()

  tokens: 3 completion | finish: stop
LLM ready: nvidia/nemotron-3-ultra-550b-a55b:free | model arithmetic: disabled


True

In [4]:
# ============================================================================
# CELL 3: Retrieval layer
# ============================================================================
import sys
sys.path.insert(0, "RAG")
import pipeline_rag as rag
for score, chunk in rag.search("Why is the automobile add-back only 25 percent?"):
    print(f"  {score:.3f}  {chunk['id']:<12} {chunk['heading'][:55]}")

print(rag.summary())

  0.485  roselle-09   7. Automobile
  0.358  roselle-10   8. Rejected add-backs
  0.346  method-05    5. Known failure modes > 5.1 Convention rates presented
  0.278  method-04    4. Add-back categories
corpus 427aa1ac3ba5 | 30 chunks (14 shared, 16 client) | model sentence-transformers/all-MiniLM-L6-v2


In [5]:
# ============================================================================
# CELL 4: Data layer — load and classify
# ============================================================================
import pandas as pd
from pathlib import Path

CSV_DIR  = Path("CSVs")
PNL_CSV  = CSV_DIR / "P&L_2022_2025_addbacks.csv"
TAX_CSV  = CSV_DIR / "Taxes_addback_analysis_all.csv"
PROD_CSV = CSV_DIR / "productivity_2022_2025_all.csv"

# The six P&L add-back categories, in the order method §4 lists them. Named once because
# three separate things need the same list: parsing, summing, and reporting.
PNL_ADDBACKS = ["Automobile", "Contributions", "Legal Fees", "Miscellaneous",
                "Travel & Entertainment", "Owner Excess Compensation"]

# Non-individual rows in the productivity export. Method §5.5: classify by record type
# before computing anything per-provider, or a headcount counts an entity row as a
# clinician. Classified in the loader so downstream code cannot forget to.
RECORD_TYPES = {
    "Roselle Dental Center": "entity",       # practice-entity row, no production
    "IC":                    "contractor",   # independent contractor dentistry, aggregated
    "Hyg Hyg":               "aggregate",    # non-attributed hygiene production
}


def parse_currency(s):
    """'$3,750.00 ' -> 3750.0 | '($6,877.36)' -> -6877.36 | 'NA' -> NaN

    Accounting parentheses mean negative. NA stays NaN rather than becoming 0: a missing
    add-back and a zero add-back are different facts, and collapsing them hides which
    years a category was absent from.
    """
    return pd.to_numeric(
        s.astype(str)
         .str.strip()
         .str.replace(r'[$,]', '', regex=True)
         .str.replace(r'^\((.*)\)$', r'-\1', regex=True)
         .replace({'NA': None, 'N/A': None, 'na': None, '': None}),
        errors="coerce")


def load_pnl(path=PNL_CSV):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for col in df.columns:
        if col != "Year":
            df[col] = parse_currency(df[col])
    df["Year"] = df["Year"].astype(int)
    return df.set_index("Year").sort_index()


def load_tax(path=TAX_CSV):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for col in df.columns:
        if col != "Year":
            df[col] = parse_currency(df[col])
    df["Year"] = df["Year"].astype(int)
    df["Tax_Addbacks"] = df["Depreciation"] + df["Section179"]
    return df.set_index("Year").sort_index()


def load_productivity(path=PROD_CSV):
    """Load the productivity export and tag every row with its record type."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df["record_type"] = df["provider_name"].map(RECORD_TYPES).fillna("individual")
    df["year"] = df["year"].astype(int)
    return df


pnl  = load_pnl()
tax  = load_tax()
prod = load_productivity()

print(f"P&L          {len(pnl)} years {list(pnl.index)}")
print(f"Tax          {len(tax)} years {list(tax.index)}  (no FY2025 return)")
print(f"Productivity {len(prod)} rows, {prod['year'].nunique()} years")
print()
print("Provider record types:")
for rt, g in prod.groupby("record_type"):
    names = sorted(g["provider_name"].unique())
    print(f"  {rt:<12} {len(names)} name(s), {len(g):>2} rows   {', '.join(names)[:70]}")

P&L          4 years [2022, 2023, 2024, 2025]
Tax          3 years [2022, 2023, 2024]  (no FY2025 return)
Productivity 36 rows, 4 years

Provider record types:
  aggregate    1 name(s),  3 rows   Hyg Hyg
  contractor   1 name(s),  3 rows   IC
  entity       1 name(s),  3 rows   Roselle Dental Center
  individual   7 name(s), 27 rows   Emily Hawkins, RDH, Farah Rahman, RDH, Harvey Seybold, DDS, Jacqueline


In [6]:
# ============================================================================
# CELL 5: Reconciliation — verify the CSVs before anything reads them
# ============================================================================

def reconcile(pnl, tax, tol=0.02):
    """Check each CSV against its own internal arithmetic.

    Two independent identities, both of which must hold for every year:
      Net                          == Credit - Debit
      Total Discretionary Earnings == Net + the six P&L add-backs

    The second is the one that matters. That column is P&L-only by convention, so a year
    that fails by exactly the tax add-back amount means tax figures were folded into a
    P&L total -- which double-counts the moment anything sums the two files.

    tol is 2 cents rather than 1: FY2025 carries a known one-cent rounding difference
    inherited from the Decision Record, and a tolerance that flags it every run trains
    you to ignore the check.
    """
    rows, failures = [], []

    for year in pnl.index:
        r = pnl.loc[year]

        net_calc  = r["Credit"] - r["Debit"]
        net_delta = net_calc - r["Net"]

        addbacks   = r[PNL_ADDBACKS].sum()      # NaN skipped: a missing line is not an add-back
        total_calc = r["Net"] + addbacks
        tot_delta  = total_calc - r["Total Discretionary Earnings"]

        rows.append({"Year": year,
                     "Net": r["Net"],
                     "P&L add-backs": addbacks,
                     "Total (file)": r["Total Discretionary Earnings"],
                     "Total (calc)": total_calc,
                     "delta": tot_delta})

        if abs(net_delta) > tol:
            failures.append(f"  {year} Net != Credit - Debit by {net_delta:,.2f}")
        if abs(tot_delta) > tol:
            hint = ""
            if year in tax.index and abs(abs(tot_delta) - tax.loc[year, "Tax_Addbacks"]) < tol:
                hint = f"  <- equals the {year} tax add-back; tax figures folded into a P&L total"
            failures.append(f"  {year} Total != Net + add-backs by {tot_delta:,.2f}{hint}")

    report = pd.DataFrame(rows).set_index("Year")
    print(report.to_string(float_format=lambda v: f"{v:,.2f}"))

    if failures:
        print("\nRECONCILIATION FAILED")
        print("\n".join(failures))
        raise ValueError("CSV does not reconcile — fix the file before proceeding")

    print("\nReconciles to the cent, all years.")
    return report


def normalized_earnings(pnl, tax):
    """Net income + P&L add-backs + tax add-backs, per method §3.

    Tax add-backs come from the return, not the P&L, so a year without a return carries
    none -- FY2025 is genuinely P&L-only, not missing data.
    """
    out = pd.DataFrame(index=pnl.index)
    out["Net income"]    = pnl["Net"]
    out["P&L add-backs"] = pnl[PNL_ADDBACKS].sum(axis=1)
    out["Tax add-backs"] = tax["Tax_Addbacks"].reindex(pnl.index).fillna(0.0)
    out["Normalized earnings"] = out.sum(axis=1)
    return out


reconcile(pnl, tax)

print()
norm = normalized_earnings(pnl, tax)
print(norm.to_string(float_format=lambda v: f"{v:,.2f}"))
print(f"\nFour-year total: {norm['Normalized earnings'].sum():,.2f}")

             Net  P&L add-backs  Total (file)  Total (calc)  delta
Year                                                              
2022   -6,877.36      14,898.46      8,021.10      8,021.10  -0.00
2023  -62,982.50      20,515.59    -42,466.91    -42,466.91   0.00
2024 -112,623.23      13,919.73    -98,703.50    -98,703.50   0.00
2025  -29,284.43       2,118.38    -27,166.04    -27,166.05  -0.01

Reconciles to the cent, all years.

      Net income  P&L add-backs  Tax add-backs  Normalized earnings
Year                                                               
2022   -6,877.36      14,898.46      14,094.00            22,115.10
2023  -62,982.50      20,515.59      18,588.00           -23,878.91
2024 -112,623.23      13,919.73       7,685.00           -91,018.50
2025  -29,284.43       2,118.38           0.00           -27,166.05

Four-year total: -119,948.36


In [7]:
# ============================================================================
# CELL 6: Provider metrics — deterministic, per-provider
# ============================================================================
import re


def audit_productivity(df=None):
    """Flag rows where production / patients_seen disagrees with the stated avg_visit_fee.

    The three columns are redundant by construction, so a disagreement means one of them
    is wrong. Which one is not determinable from the file -- report, do not repair.
    """
    d = (prod if df is None else df).copy()
    d["calc_fee"] = (d["production"] / d["patients_seen"]).where(d["patients_seen"] > 0)
    bad = d[(d["calc_fee"] - d["avg_visit_fee"]).abs() > 0.02]
    return bad[["year", "provider_name", "patients_seen", "production",
                "avg_visit_fee", "calc_fee"]]


def resolve_provider(query, df=None):
    """Map a name fragment to an exact provider_name.

    Returns (name, note). name is None when nothing matches or the match is ambiguous;
    note says which. Credentials vary in the export ('DMI', 'RI'), so matching is on
    name tokens rather than the full string.
    """
    df = prod if df is None else df
    names = sorted(df["provider_name"].unique())
    q = query.lower().strip()

    exact = [n for n in names if n.lower() == q]
    if exact:
        return exact[0], ""

    contained = [n for n in names if n.lower() in q or q in n.lower()]
    if len(contained) == 1:
        return contained[0], ""

    tokens = [t for t in re.findall(r"[a-z]+", q) if len(t) > 2]
    scored = []
    for n in names:
        parts = set(re.findall(r"[a-z]+", n.lower()))
        hits = sum(1 for t in tokens if t in parts)
        if hits:
            scored.append((hits, n))
    if not scored:
        return None, f"no provider matching {query!r}"

    best = max(h for h, _ in scored)
    winners = [n for h, n in scored if h == best]
    if len(winners) > 1:
        return None, f"ambiguous: {', '.join(winners)}"
    return winners[0], ""


def provider_facts(name):
    """Per-year facts for one provider, plus the caveats that apply to reading them."""
    resolved, note = resolve_provider(name)
    if resolved is None:
        return {"provider": None, "error": note}

    g = prod[prod["provider_name"] == resolved].set_index("year").sort_index()
    rec = g["record_type"].iloc[0]

    table = g[["production", "collection", "patients_seen", "adjustments",
               "avg_visit_fee"]].copy()
    # Recomputed alongside the stated figure rather than replacing it: where they differ,
    # the row is unreliable and the caveat says so.
    table["fee_check"] = (table["production"] / table["patients_seen"]).where(
        table["patients_seen"] > 0)

    caveats = []
    suspect = (table["fee_check"] - table["avg_visit_fee"]).abs() > 0.02
    if suspect.any():
        yrs = ", ".join(str(y) for y in table.index[suspect])
        caveats.append(
            f"{yrs}: production / patients_seen disagrees with the stated avg_visit_fee. "
            "One of the three columns is wrong for that year; totals and averages "
            "spanning it are unreliable.")

    if rec != "individual":
        caveats.append(
            f"'{resolved}' is a {rec} record, not an individual provider (method §5.5). "
            "It must not be counted in per-provider averages or headcounts.")

    tot_prod, tot_pat = table["production"].sum(), table["patients_seen"].sum()

    # Collection is attributed on a different basis than production in this export, so a
    # per-provider rate is not computable. Method §5.5.
    if tot_prod > 0:
        ratio = table["collection"].sum() / tot_prod
        if ratio < 0.05 or ratio > 1.2:
            caveats.append(
                "Collection is not attributed to this provider on the same basis as "
                "production, so a per-provider collection rate is not computable. "
                "Use the practice-level rate.")

    return {
        "provider": resolved,
        "role": g["service_role"].iloc[0],
        "record_type": rec,
        "years": list(table.index),
        "table": table,
        "total_production": tot_prod,
        "total_patients": int(tot_pat),
        # Weighted, not an unweighted mean of yearly rates: that would weight a 7-visit
        # year the same as a 2,239-visit year.
        "avg_visit_fee_overall": (tot_prod / tot_pat) if tot_pat else None,
        "reliable": not suspect.any() and rec == "individual",
        "caveats": caveats,
    }


def provider_list(individuals_only=True):
    """Named clinicians, or every record in the export."""
    df = prod[prod["record_type"] == "individual"] if individuals_only else prod
    return sorted(df["provider_name"].unique())

def provider_fact_lines():
    """One line per provider, name adjacent to every figure.

    A wide pivot asks the model to hold a column heading in mind while reading four rows
    beneath it, which is where the misattribution came from — right figures, wrong
    column. A line per provider makes the association local and unmissable.
    """
    share = production_share("Dentist")
    doll  = prod[prod.service_role == "Dentist"].pivot_table(
        index="year", columns="provider_name", values="production", aggfunc="sum").fillna(0)

    owners = {"Harvey Seybold, DDS", "Peter S. Chang, DDS"}
    lines = []
    for name in share.columns:
        rt   = prod.loc[prod.provider_name == name, "record_type"].iloc[0]
        role = ("owner dentist" if name in owners else
                "non-owner dentist" if rt == "individual"
                else f"{rt} record, not an individual")
        pct = "; ".join(f"{y} {share.loc[y, name]:.1f}%" for y in share.index)
        amt = "; ".join(f"{y} {doll.loc[y, name]:,.2f}" for y in doll.index)
        lines.append(f"{name} — {role}. Share of dentist production: {pct}. "
                     f"Production in dollars: {amt}.")
    return lines


def _reps(v):
    """Every rounding of a figure a reader might see, so a quoted 44.5 matches 44.48."""
    v = float(v)
    return {round(v, 2), round(v, 1), round(v, 0)}


def attribution_errors(text, tol=0.005):
    """Flag figures stated next to the wrong provider's name.

    check() cannot catch this class: the numbers are all present in the source, just
    bound to the wrong person.

    Deliberately narrow. A figure is tested only when exactly one provider is named in
    its sentence — in "Chang drew X against Seybold's Y and produced 1.9%", nothing local
    says which name the 1.9% attaches to, and guessing produces false failures on correct
    prose. Missing those is better than crying wolf on them.
    """
    share = production_share("Dentist")
    doll  = prod[prod.service_role == "Dentist"].pivot_table(
        index="year", columns="provider_name", values="production", aggfunc="sum").fillna(0)

    owned = {}
    for n in share.columns:
        s = set()
        for v in list(share[n]) + list(doll[n]):
            s |= _reps(v)
        owned[n] = s

    surnames = {n: n.split(",")[0].split()[-1] for n in share.columns}
    problems = []

    for sentence in re.split(r'(?<=[.;])\s+', text):
        named = [n for n, sur in surnames.items()
                 if re.search(r'\b' + re.escape(sur) + r'\b', sentence)]
        if len(named) != 1:            # zero or ambiguous — not decidable locally
            continue
        name = named[0]
        for tok in re.findall(r'\d[\d,]*(?:\.\d+)?', sentence):
            try:
                val = round(float(tok.replace(",", "")), 2)
            except ValueError:
                continue
            if 1900 < val < 2100:
                continue
            if any(abs(val - o) < tol for o in owned[name]):
                continue
            others = [n for n, s in owned.items()
                      if n != name and any(abs(val - o) < tol for o in s)]
            if others:
                problems.append(f"{tok} attributed to {name}; belongs to {', '.join(others)}")
    return problems


issues = audit_productivity()
print(f"productivity audit: {len(issues)} inconsistent row(s) of {len(prod)}")
if len(issues):
    print(issues.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

print(f"\nindividuals: {len(provider_list())} | all records: {len(provider_list(False))}")
for q in ["Emily Hawkins, RDH", "Seybold", "Chang", "Bob Smith"]:
    n, note = resolve_provider(q)
    print(f"  {q:<20} -> {n or '(none)'} {note}")

productivity audit: 1 inconsistent row(s) of 36
 year      provider_name  patients_seen  production  avg_visit_fee  calc_fee
 2024 Emily Hawkins, RDH             77   61,985.84         164.42    805.01

individuals: 7 | all records: 10
  Emily Hawkins, RDH   -> Emily Hawkins, RDH 
  Seybold              -> Harvey Seybold, DDS 
  Chang                -> Peter S. Chang, DDS 
  Bob Smith            -> (none) no provider matching 'Bob Smith'


In [8]:
# ============================================================================
# CELL 7: Practice metrics — financial and production, practice-wide
# ============================================================================

PARTIAL_YEARS = {2025: "~149 days"}      # method §5.4: not annualized, not comparable

MEASURE = "normalized owner earnings"
MEASURE_NOTE = (
    "Owner compensation is normalized to a market rate and only the excess is added back, "
    "so this is normalized owner earnings, not SDE as the term is used in practice sales "
    "(method §1). The delivered files are labeled SDE.")


def earnings_by_year():
    """Normalized owner earnings per year, with its components. Method §3."""
    out = normalized_earnings(pnl, tax)
    out["partial year"] = [PARTIAL_YEARS.get(y, "") for y in out.index]
    return out


def addback_components(year=None):
    """The six P&L add-backs plus the two tax add-backs, by year or across all years.

    Kept separate by source: they come from different documents, and only the P&L file
    is competent for the former (method §2).
    """
    cols = pnl[PNL_ADDBACKS]
    out = (cols if year is None else cols.loc[[year]]).copy()
    out["Depreciation"] = tax["Depreciation"].reindex(out.index)
    out["Section 179"]  = tax["Section179"].reindex(out.index)
    return out


def practice_totals():
    """Production, collection and visits by year, over every record in the export.

    Practice-level only. Collection is attributed on a different basis than production
    per provider, so the rate is meaningful here and not per clinician (method §5.5).
    """
    g = prod.groupby("year").agg(production=("production", "sum"),
                                 collection=("collection", "sum"),
                                 visits=("patients_seen", "sum"))
    g["collection rate %"] = g["collection"] / g["production"] * 100
    g["visits"] = g["visits"].astype(int)
    g["partial year"] = [PARTIAL_YEARS.get(y, "") for y in g.index]
    return g


def production_share(role="Dentist"):
    """Share of production within one service role, by provider and year.

    Includes non-individual records where they carry production: the contractor line is
    real production, and dropping it would overstate every other share. It is labeled,
    not hidden. Note a 0.0 means a row with no production; an absent provider means no
    row at all.
    """
    d = prod[prod["service_role"] == role]
    piv = d.pivot_table(index="year", columns="provider_name",
                        values="production", aggfunc="sum").fillna(0)
    return piv.div(piv.sum(axis=1), axis=0) * 100


def valuation_range(*_, **__):
    """Deliberately not computed. Three reasons, all structural rather than temporary."""
    e = earnings_by_year()["Normalized earnings"]
    return {
        "computed": False,
        "reasons": [
            MEASURE_NOTE + " A multiple applied to this measure does not answer what a "
            "buyer's multiple answers; the figure must be recomputed under a full-add-back "
            "convention first.",
            "Full add-back requires the owner's total compensation, which is not in these "
            "files — only the excess over benchmark is. The converted figures are in the "
            "engagement record.",
            f"{int((e < 0).sum())} of {len(e)} years are negative; a multiple applied to a "
            "negative figure yields a negative valuation, which is not a result.",
        ],
        "earnings_by_year": e.to_dict(),
    }


print("Normalized owner earnings")
print(earnings_by_year().to_string(float_format=lambda v: f"{v:,.2f}"))
print("\nPractice totals")
print(practice_totals().to_string(float_format=lambda v: f"{v:,.2f}"))
print("\nDentist production share (%)")
print(production_share("Dentist").to_string(float_format=lambda v: f"{v:.1f}"))

Normalized owner earnings
      Net income  P&L add-backs  Tax add-backs  Normalized earnings partial year
Year                                                                            
2022   -6,877.36      14,898.46      14,094.00            22,115.10             
2023  -62,982.50      20,515.59      18,588.00           -23,878.91             
2024 -112,623.23      13,919.73       7,685.00           -91,018.50             
2025  -29,284.43       2,118.38           0.00           -27,166.05    ~149 days

Practice totals
       production   collection  visits  collection rate % partial year
year                                                                  
2022 1,278,542.88 1,134,654.92    6787              88.75             
2023 1,251,957.50 1,097,174.27    6963              87.64             
2024 1,267,037.45 1,085,020.82    6377              85.63             
2025   473,576.39   376,373.71    2359              79.47    ~149 days

Dentist production share (%)
provider_name  

In [14]:
# ============================================================================
# CELL 8: Report sections — decide what the data supports, before any prose
# ============================================================================

def _fmt(df, dp=2):
    return df.to_string(float_format=lambda v: f"{v:,.{dp}f}")


def sec_measure():
    """Which measure was computed, and what it is not. Method §1."""
    e = earnings_by_year()["Normalized earnings"]
    return {"figures": _fmt(e.to_frame()),
            "notes": [MEASURE_NOTE],
            "query": "choosing the measure normalized owner earnings versus SDE labeling"}


def sec_financials():
    """Collection rate only. Production, collection and visits are in the rendered
    table; handing them to the model produced a narrated walk-through and a false
    trend claim ('production declined') from figures that show a rise.
    """
    t = practice_totals()
    lines = [f"FY{y}: collection rate {t.loc[y, 'collection rate %']:.2f}%"
             + (f" (partial year, {PARTIAL_YEARS[y]})" if y in PARTIAL_YEARS else "")
             for y in t.index]

    return {"figures": "\n".join(lines),
            "notes": [f"FY{y} is a partial year ({d}); figures are not annualized and "
                      f"not comparable to full years (method §5.4)."
                      for y, d in PARTIAL_YEARS.items()],
            "query": "production collection partial year proration"}


def sec_earnings():
    return {"figures": _fmt(earnings_by_year()),
            "notes": ["Tax-side add-backs come from the return, so a year without a "
                      "return carries none (method §2)."],
            "query": "formula net income add-backs depreciation section 179"}


def sec_addbacks():
    comp = addback_components()
    zero = [c for c in comp.columns if comp[c].fillna(0).eq(0).all()]

    # NaN means "no line in the source", which is not zero. Rendered as a dash so the
    # prompt cannot read it as an amount, with the reason stated rather than implied.
    shown = comp.copy()
    figures = shown.to_string(float_format=lambda v: f"{v:,.2f}", na_rep="—")

    notes = ["A dash means the category has no line in the source document for that "
             "year, which is not the same as an amount of zero."]
    if any(comp[c].isna().any() for c in ("Depreciation", "Section 179")):
        notes.append("FY2025 has no depreciation or Section 179 because no FY2025 tax "
                     "return was available (method §2).")
    if zero:
        notes.append("Categories with no amount in any year: " + ", ".join(zero))

    return {"figures": figures, "notes": notes,
            "query": "add-back categories accepted rejected estimated ledger source documents tax return"}


def sec_providers():
    share  = production_share("Dentist")
    issues = audit_productivity()
    notes  = []
    nonind = sorted(prod.loc[prod.record_type != "individual", "provider_name"].unique())
    if nonind:
        notes.append(f"Non-individual records present: {', '.join(nonind)}. These are "
                     "record types, not clinicians, and are excluded from headcounts and "
                     "per-provider averages (method §5.5).")
    for _, r in issues.iterrows():
        notes.append(f"{r.provider_name} FY{int(r.year)}: production / patients_seen "
                     f"({r.production / r.patients_seen:,.2f}) disagrees with the stated "
                     f"avg_visit_fee ({r.avg_visit_fee:,.2f}). Figures spanning that year "
                     "are unreliable.")
    notes.append("Collection is attributed on a different basis than production per "
                 "provider, so collection rate is reported practice-wide only.")
    return {"figures": "\n".join(provider_fact_lines()),
            "notes": notes,
            "query": "provider record types non-owner clinician concentration"}


def sec_valuation():
    v = valuation_range()
    return {"figures": "No multiple applied.",
            "notes": v["reasons"],
            "query": "measure labeling relabel recompute before presenting to a buyer"}


def sec_open_items():
    """Non-owner compensation and the health-insurance estimate. The owners-list note
    and the dollar pivot both pulled §5.3 multi-owner material into this section,
    which duplicated and then contradicted the measure section.
    """
    share  = production_share("Dentist")
    owners = ["Harvey Seybold, DDS", "Peter S. Chang, DDS"]
    ind    = set(prod.loc[prod.record_type == "individual", "provider_name"])
    non_owner = [c for c in share.columns if c in ind and c not in owners]

    lines = [f"{n}: " + "; ".join(f"FY{y} {share.loc[y, n]:.1f}%" for y in share.index)
             + " of dentist production" for n in non_owner]

    return {
        "figures": "\n".join(lines),
        "notes": [
            f"{', '.join(non_owner)}: non-owner clinician, not in payroll account 6575.",
            "No supplied file contains compensation for any non-owner clinician.",
            "Health insurance was excluded from the calculation as an unsubstantiated "
            "estimate rather than substantiated from the ledger.",
        ],
        "query": "non-owner clinician concentration unsubstantiated estimate "
                 "substantiate from the ledger",
    }


SECTION_SPEC = [
    ("measure",    "Measure and labeling",        sec_measure),
    ("financials", "Key financial indicators",    sec_financials),
    ("earnings",   "Normalized earnings by year", sec_earnings),
    ("addbacks",   "Add-back detail",             sec_addbacks),
    ("providers",  "Provider production",         sec_providers),
    ("valuation",  "Valuation multiples",         sec_valuation),
    ("open_items", "Open items",                  sec_open_items),
]


def build_sections():
    """Run every section builder. A section survives if it has figures or notes.

    Deciding this in code rather than in the prompt is the point: a template that asks
    the model to fill a heading gets a filled heading whether or not the data supports one.
    """
    out = []
    for key, title, fn in SECTION_SPEC:
        try:
            s = fn()
        except Exception as e:
            print(f"  SKIP {key}: {type(e).__name__}: {e}")
            continue
        s.update(key=key, title=title)
        s["supported"] = bool(s["figures"].strip() or s["notes"])
        out.append(s)
    return out


sections = build_sections()
for s in sections:
    mark = "" if s["supported"] else "  (no data — carried by the corpus alone)"
    print(f"  {s['key']:<11} {s['title']:<28} {len(s['notes'])} note(s){mark}")

  measure     Measure and labeling         1 note(s)
  financials  Key financial indicators     1 note(s)
  earnings    Normalized earnings by year  1 note(s)
  addbacks    Add-back detail              2 note(s)
  providers   Provider production          3 note(s)
  valuation   Valuation multiples          3 note(s)
  open_items  Open items                   3 note(s)


In [10]:
# ============================================================================
# CELL 9: Guardrails
# ============================================================================

def report_check(rep):
    """One-line verdict plus whatever failed. Display only — the checks live in rag."""
    print(f"GUARDRAIL : {'PASS' if rep['pass'] else 'FAIL'}")
    for key in ("fabricated_citations", "malformed_citations", "misattributed_figures",
                "unsupported_numbers", "leaked_in_refusal", "uncited"):
        if rep.get(key):
            print(f"    {key}: {rep[key]}")


# Confirm the checks fire rather than passing everything by default.
_bad = rag.check("The FY2022 excess was 41,200.00 [roselle-99].", ["roselle-06"])
assert not _bad["pass"] and _bad["fabricated_citations"] == ["roselle-99"]
print("guardrails live |", rag.summary())

guardrails live | corpus 427aa1ac3ba5 | 30 chunks (14 shared, 16 client) | model sentence-transformers/all-MiniLM-L6-v2


In [15]:
# ============================================================================
# CELL 10: Section generation — retrieve, prompt, check
# ============================================================================

SECTION_SYSTEM = """You write one section of a dental practice valuation report.

You are given FIGURES (computed from the practice's source files), NOTES (facts about
those figures and about what the files do not contain), and SOURCES (numbered excerpts
from the firm's methodology and this engagement's record).

FIGURES and NOTES are facts. SOURCES are the rules and findings that explain them.

Rules:
1. Quote every figure exactly as it appears in FIGURES or NOTES. Never round, restate to
   fewer decimal places, recalculate, or introduce a number that is not given.
2. Cite every methodological claim inline as [source-id]. Any statement about why a
   treatment is correct, what a rule requires, or what a finding means must carry a
   citation to the source it rests on.
3. If SOURCES do not support a claim, do not make it. Do not fill gaps from general
   knowledge about valuation or accounting.
4. Draw conclusions from the rules, do not repeat the notes. A note states a fact; your
   job is to say what the cited rule makes of it. Never state the same fact twice.
5. Do not add recommendations, next steps, or follow-up suggestions.
6. Write plain professional prose, two to four short paragraphs. Do not write a heading.
7. The FIGURES table is printed in the report directly above your text. Do not reproduce
   it, in prose or as a table, and do not walk through it year by year. Name a figure
   only where a cited rule bears on it — at most three or four in the whole section.
8. Cite only in the form [source-id], using an id exactly as it appears in brackets at
   the head of a SOURCE — for example [method-04]. The excerpts contain provenance marks
   and cross-references such as [doc · DR §4] or "method §2"; these are part of the
   source text, not citation ids, and must never be used as citations. Facts from FIGURES
   and NOTES need no citation at all — state them plainly, with no bracket of any kind.
   Never write [FIGURES] or [NOTES].
9. Output the finished section only. No analysis, no plan, no commentary about the task.
   The first words of your reply are the first words of the report."""

ARITHMETIC_CLAUSE = """
7. You may compute simple derived figures (differences, percentages, totals) from the
   figures given. State the inputs whenever you do."""

_UNCITABLE = re.compile(r'\s*\[(?:NOTES|FIGURES)\]')


def clean_section(text):
    """Remove [NOTES] and [FIGURES] labels the model attaches to data-derived facts.

    These are the only two uncitable sources in the prompt, and the model reliably labels
    facts drawn from them despite three prompt revisions telling it not to. Stripping is
    safe because the labels carry no information — the sentence is a plain statement of
    data either way. A real citation to a chunk id is never touched.
    """
    return _UNCITABLE.sub("", text)


def section_prompt(sec, hits):
    """Assemble one section's prompt. The chunk breadcrumb is dropped from the body
    because it is already the source label, and repeating it wastes context."""
    sources = "\n\n".join(
        f"[{c['id']}] {c['heading']}\n{c['text'].split(chr(10) * 2, 1)[-1]}"
        for _, c in hits)
    notes = "\n".join(f"- {n}" for n in sec["notes"]) or "(none)"
    return (f"SECTION: {sec['title']}\n\n"
            f"FIGURES:\n{sec['figures'].strip() or '(none)'}\n\n"
            f"NOTES:\n{notes}\n\n"
            f"SOURCES:\n\n{sources}\n\n"
            f"Write the body of this section.")


def generate_section(sec, k=4, max_tokens=1600):
    """Retrieve for this section, generate its prose, check the result."""
    hits   = rag.search(sec["query"], k=k)
    system = SECTION_SYSTEM + (ARITHMETIC_CLAUSE if ALLOW_MODEL_ARITHMETIC else "")
    text   = clean_section(call_llm(system, section_prompt(sec, hits),
                                    max_tokens=max_tokens))

    # Figures and notes join the retrieved chunks as supporting context: without them
    # every correctly-quoted CSV figure would flag as unsupported.
    context = sec["figures"] + "\n" + "\n".join(sec["notes"])
    rep = rag.check(text, [c["id"] for _, c in hits], extra_context=context)

    # Attribution is checked here rather than in rag.check(): it needs the provider data,
    # which the corpus does not have.
    if sec["key"] in ("providers", "open_items"):
        misattributed = attribution_errors(text)
        if misattributed:
            rep["misattributed_figures"] = misattributed
            rep["pass"] = False

    return {"key": sec["key"], "title": sec["title"], "text": text,
            "retrieved": [(round(s, 3), c["id"]) for s, c in hits], "check": rep}


def show_section(out):
    assert out is not None, "generate_section returned None — check its return statement"
    print(f"{'=' * 78}\n## {out['title']}\n")
    print(f"RETRIEVED : {out['retrieved']}")
    report_check(out["check"])
    print()
    print(rag.render(out["text"]))

def regen(*keys):
    """Regenerate named sections in place. One call each.

    The full loop costs seven calls; during tuning almost every fix touches one
    section, and the daily limit is 50.
    """
    by_key = {s["key"]: s for s in sections}
    for i, o in enumerate(outputs):
        if o["key"] in keys:
            outputs[i] = generate_section(by_key[o["key"]])
            show_section(outputs[i])
            time.sleep(3)

    fails = [o["key"] for o in outputs if not o["check"]["pass"]]
    print(f"\n{len(outputs) - len(fails)}/{len(outputs)} sections passed"
          + (f" | failed: {fails}" if fails else ""))

In [16]:
regen("financials", "valuation", "open_items")

  retryable (429), waiting 15s — {"error":{"message":"Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day



KeyboardInterrupt



In [12]:
outputs = []
for sec in sections:
    out = generate_section(sec)
    outputs.append(out)
    show_section(out)
    time.sleep(3)

fails = [o["key"] for o in outputs if not o["check"]["pass"]]
print(f"\n{len(outputs) - len(fails)}/{len(outputs)} sections passed"
      + (f" | failed: {fails}" if fails else ""))

  tokens: 394 completion | finish: stop
## Measure and labeling

RETRIEVED : [(0.664, 'roselle-01'), (0.633, 'method-01'), (0.461, 'roselle-08'), (0.417, 'method-07')]
GUARDRAIL : PASS

The engagement was prepared for the owners to determine whether the practice was profitable after the owner's labor is fairly compensated, which selects normalized owner earnings rather than SDE [Roselle: Engagement purpose]. Under the chosen measure, owner compensation is normalized to a market rate and only the excess is added back, whereas SDE would add back the working owner's entire compensation [Method: Choosing the measure before computing anything]. The delivered files are labeled "SDE," but the measure actually computed is normalized owner earnings, producing figures of 22,115.10 for 2022, −23,878.91 for 2023, −91,018.50 for 2024, and −27,166.05 for 2025. Under a full-add-back convention the same ledger reads approximately 216,000 for FY2022 and approximately 165,000 for FY2023, a difference la

In [13]:
# ============================================================================
# CELL 11: Report assembly — tables beside the prose
# ============================================================================

def df_to_markdown(df, dp=2, index_name=None):
    """Render a DataFrame as a markdown table.

    pandas to_string() is fixed-width text: fine in a terminal, unreadable once the
    report is markdown. Numerics are right-aligned and missing values render as an em
    dash, matching the convention the prose refers to.
    """
    d    = df.copy()
    idx  = index_name or (d.index.name or "")
    head = [str(idx)] + [str(c) for c in d.columns]

    rows = []
    for i, r in d.iterrows():
        cells = []
        for v in r:
            if isinstance(v, float) and pd.isna(v):
                cells.append("—")
            # Integers before floats: bool and int both satisfy the float branch's intent,
            # and a visit count rendered as 6,787.00 reads as a currency amount.
            elif isinstance(v, (int, np.integer)) and not isinstance(v, bool):
                cells.append(f"{int(v):,}")
            elif isinstance(v, (float, np.floating)):
                cells.append(f"{v:,.{dp}f}")
            else:
                cells.append(str(v) if str(v).strip() else "")
        rows.append([str(i)] + cells)

    align = ["| :--- "] + [" ---: " for _ in d.columns]
    return "\n".join(["| " + " | ".join(head) + " |", "|".join(align) + "|"]
                     + ["| " + " | ".join(r) + " |" for r in rows])


def dentist_dollars():
    d = prod[(prod.service_role == "Dentist") & (prod.record_type == "individual")]
    return d.pivot_table(index="year", columns="provider_name",
                         values="production", aggfunc="sum").fillna(0)


# Which table belongs beside which section's prose. valuation maps to None — the absence
# of a figure is the finding there.
FRAMES = {
    "measure":    (lambda: earnings_by_year()[["Normalized earnings"]],     2),
    "financials": (practice_totals,                                          2),
    "earnings":   (lambda: earnings_by_year().drop(columns="partial year"),  2),
    "addbacks":   (addback_components,                                       2),
    "providers":  (lambda: production_share("Dentist"),                      1),
    "valuation":  (None,                                                     0),
    "open_items": (dentist_dollars,                                          2),
}

TABLE_CAPTION = {
    "measure":    "Normalized owner earnings by fiscal year",
    "financials": "Production, collections and visits by fiscal year",
    "earnings":   "Components of normalized owner earnings",
    "addbacks":   "Add-backs by category and fiscal year",
    "providers":  "Dentist production share (%)",
    "open_items": "Dentist production, dollars (named individuals only)",
}


def render_report(outputs, title="Valuation Analysis", subtitle=""):
    """Interleave heading, table, and prose for every section.

    The prompt tells the model the table is printed above its text. Until this function
    existed that was false, and the prose compensated by reciting figures the reader
    could not see.
    """
    parts = [f"# {title}"]
    if subtitle:
        parts.append(f"*{subtitle}*")

    for out in outputs:
        parts.append(f"\n## {out['title']}")
        fn, dp = FRAMES.get(out["key"], (None, 2))
        if fn is not None:
            if out["key"] in TABLE_CAPTION:
                parts.append(f"**{TABLE_CAPTION[out['key']]}**\n")
            parts.append(df_to_markdown(fn(), dp=dp))
            parts.append("")
        parts.append((rag.render(out["text"]).strip()))

    return "\n".join(parts)

from IPython.display import Markdown, display

report_md = render_report(
    outputs,
    title="Roselle Dental Center, P.C. — Normalized Earnings Analysis",
    subtitle="FY2022–FY2025 · prepared for the owners")  # clean, real $ signs

# The file is correct as-is.
Path("Roselle_Report.md").write_text(report_md, encoding="utf-8")


def display_report(md):
    """Show the report in Jupyter without MathJax eating currency.

    Two defenses because the class name is version-dependent: MathJax 2 honors
    tex2jax_ignore, MathJax 3 (JupyterLab 3+, Notebook 7) honors mathjax_ignore. Wrapping
    each $ in its own element breaks delimiter pairing regardless, since MathJax needs
    both delimiters in one text node.
    """
    from IPython.display import HTML, display
    import markdown as md_lib
    html = md_lib.markdown(md, extensions=["tables"])
    html = html.replace("$", "<span>$</span>")
    display(HTML(f'<div class="mathjax_ignore tex2jax_ignore">{html}</div>'))


display_report(report_md)